# Observation Map

Analyze species occurrence data from FinBIF/GBIF Darwin Core Archives.

## Overview

This notebook downloads biodiversity occurrence data from a FinBIF collection and analyzes locations for the 5 most commonly observed species. It helps answer questions like:
- Which species are most frequently observed?
- Where geographically are these species distributed?

**Visualizations:**
- Map visualization showing species occurrence locations colored by species

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import os
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx


## Step 2: Configure Collection and Paths

In [ ]:
# Set the collection ID from https://laji.fi/en/theme/dataset-metadata
collection_id = "HR.110"  # Example: HR.7580 (change this to your collection. Available ones: https://gbif.laji.fi/list)

# Define paths
url = f"https://gbif.laji.fi/archive/{collection_id}"
data_folder = "data"
os.makedirs(data_folder, exist_ok=True)
zip_path = os.path.join(data_folder, f"{collection_id}.zip")
extract_path = os.path.join(data_folder, f"{collection_id}_extracted")

print(f"Collection ID: {collection_id}")
print(f"Download URL: {url}")

## Step 3: Download and Extract Archive

In [ ]:
# Download the archive if it doesn't exist
if not os.path.exists(zip_path):
    print(f"Downloading archive from {url}...")
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(zip_path, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded successfully: {zip_path}")
    except Exception as e:
        print(f"Error downloading: {e}")
        print("It is possible that this ID is not valid or available from GBIF.")
        raise
else:
    print(f"Using existing archive: {zip_path}")

# Extract the archive
print(f"Extracting to {extract_path}...")
os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Extraction complete.")

# Remove the zip archive
os.remove(zip_path)
print(f"Removed archive: {zip_path}")

## Step 4: Load Occurrence Data

In [ ]:
# Find occurrence.txt in the extracted archive
occurrences_file = None
for root, dirs, files in os.walk(extract_path):
    if 'occurrence.txt' in files:
        occurrences_file = os.path.join(root, 'occurrence.txt')
        break

if not occurrences_file:
    raise FileNotFoundError(f"occurrence.txt not found in {extract_path}")

# Load the data
df = pd.read_csv(occurrences_file, sep='\t', low_memory=False)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.decimalLongitude, df.decimalLatitude), crs="EPSG:4326")
print(f"\nLoaded {len(gdf)} occurrence records: ")
print(gdf.head().to_string())

## Step 5: Identify Top Species

In [ ]:
top_n_species = 5  # Number of top species to analyze

# Count observations per species
species_counts = gdf[gdf['verbatimIdentification'].notna()].groupby('verbatimIdentification').size().reset_index(name='count')
species_counts = species_counts.sort_values('count', ascending=False)

print(f"Top {top_n_species} most common species:\n")
print(gdf['verbatimIdentification'].value_counts().head().to_string(index=True))

# Get top species list
top_species_list = species_counts.head(top_n_species)['verbatimIdentification'].tolist()

# Filter data to top specie
gdf_top = gdf[gdf['verbatimIdentification'].isin(top_species_list)]
print(f"\nRecords for top {top_n_species} species: {len(gdf_top)}")


## Step 6: Map Species Geographic Distribution

In [ ]:

# Plot using geopandas built-in categorical coloring
fig, ax = plt.subplots(figsize=(12, 10))
gdf_top.plot(column='verbatimIdentification', ax=ax, alpha=0.6, markersize=30, 
                 edgecolor='black', linewidth=0.5, legend=True, cmap='tab10')

# Add OSM basemap
ctx.add_basemap(ax, crs=gdf_top.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=5, alpha=0.5)

# Formatting
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title(f'Geographic Distribution of {top_n_species} Most Common Species (Finland)\nCollection: {collection_id}', 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nMap displayed with {len(gdf_top)} occurrence records in Finland.")
print("\nSpecies distribution summary:")
print(gdf_top['verbatimIdentification'].value_counts())